<p style="height:100px;line-height:100px;background-color:cyan;text-align:center;font-size:52px;font-family:serif;font-weight:bold;color:black;border-radius:12px;">
    Loan Credit
</p>

# **1.**
<p style="height:80px;line-height:80px;background-color:lime;padding-left:12px;font-size:36px;font-family:serif;font-weight:bold;color:black;border:1px outset gold;border-radius:10px;">
    Importing Libraries & Data
</p>

In [ ]:
import polars as pl
import polars.selectors as cs
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train_df = pl.read_csv('Datasets/credit_train.csv', ignore_errors=True)

# **2.**
<p style="height:80px;line-height:80px;background-color:lime;padding-left:12px;font-size:36px;font-family:serif;font-weight:bold;color:black;border:1px outset gold;border-radius:10px;">
    Exploring Data
</p>

In [ ]:
train_df.head()

In [ ]:
print(f'Count of rows = {train_df.height}, and columns = {train_df.width}')

In [ ]:
train_df.null_count()

In [ ]:
train_df.select(
    cs.numeric()
).describe()

In [ ]:
train_df.select(
    pl.all().approx_n_unique()
)

## **2.1**
<p style="height:60px;line-height:60px;background-color:navy;padding-left:12px;font-size:32px;font-family:serif;font-weight:bold;color:white;border:1px outset gold;border-radius:8px;">
    Cleaning
</p>

In [ ]:
df_clean = train_df.with_columns(
    pl.col('Months since last delinquent').replace('NA', None)
)

**NullValues**

---

In [ ]:
df_clean.null_count()

**DuplicatesValues**

---

In [ ]:
df_clean.select(
    pl.all().approx_n_unique()
)

**Convert `Loan Status` to Numerical**

---

In [ ]:
df_clean = df_clean.with_columns(
    pl.col('Loan Status').replace(
        {
            'Charged Off' : 0,
            'Fully Paid'  : 1
        }
    ).alias('Result').cast(pl.Int64)
)

### **2.1.1**
<p style="height:50px;line-height:50px;background-color:yellow;padding-left:12px;font-size:28px;font-family:serif;font-weight:bold;color:black;border:1px outset gold;border-radius:6px;">
    Drop Missing Values
</p>

In [ ]:
df_clean = df_clean.drop(['Months since last delinquent'])

In [ ]:
df_clean = df_clean.drop_nulls()

In [ ]:
print(f'Count of rows = {df_clean.height}, and columns = {df_clean.width}')

In [ ]:
df_clean.null_count()

### **2.1.2**
<p style="height:50px;line-height:50px;background-color:yellow;padding-left:12px;font-size:28px;font-family:serif;font-weight:bold;color:black;border:1px outset gold;border-radius:6px;">
    Drop Duplicates
</p>

In [ ]:
df_clean = df_clean.unique(subset=['Loan ID'])

In [ ]:
df_clean.filter(
    pl.col('Loan ID').is_duplicated()
)

**CHECK**

---

In [ ]:
df_clean.null_count()

In [ ]:
df_clean.filter(
    pl.col('Loan ID').is_duplicated()
)

### **2.1.3**
<p style="height:50px;line-height:50px;background-color:yellow;padding-left:12px;font-size:28px;font-family:serif;font-weight:bold;color:black;border:1px outset gold;border-radius:6px;">
    Drop Unnecessary Features
</p>

In [ ]:
zeros_cols = [
    c for c in df_clean.select(cs.numeric()).columns 
    if df_clean[c].median() == 0
]

In [ ]:
for col_name in zeros_cols:
    print(f'\n ------- {col_name} -------\n')
    print(df_clean[col_name].value_counts())
    print('\n################################################\n')

In [ ]:
df_main = df_clean.drop(['Loan ID', 'Customer ID', 'Number of Credit Problems', 'Bankruptcies', 'Tax Liens'])

### **2.1.4**
<p style="height:50px;line-height:50px;background-color:yellow;padding-left:12px;font-size:28px;font-family:serif;font-weight:bold;color:black;border:1px outset gold;border-radius:6px;">
    Discover & Drop Outliers
</p>

In [ ]:
# Define Lower and Upper Bounds of Oytliers
def getOutliersBounds(df, col_name):
    q1  = df[col_name].quantile(0.25)
    q3  = df[col_name].quantile(0.75)
    iqr = q3 - q1
    
    # Define bounds
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)
    
    return lower_bound, upper_bound

In [ ]:
numeric_col = df_main.select(cs.numeric()).columns
n_cols = 4
n_rows = int(np.ceil(len(numeric_col) / n_cols))

fig, axs = plt.subplots(figsize=(12, 8), ncols=n_cols, nrows=n_rows)

for r in range(n_rows):
    for c in range(n_cols):
        ndx = c + (4 * r) +1
        try:
            lwb, upb = getOutliersBounds(df_main, numeric_col[ndx])
            data     = df_main.select(pl.col(numeric_col[ndx]))
            sns.boxplot(data=data, legend=False, ax=axs[r, c])
            axs[r, c].axhline(y=upb, color='red', linestyle='--')
            axs[r, c].axhline(y=lwb, color='red', linestyle='-.')
            axs[r, c].set_title(numeric_col[ndx])
            axs[r, c].tick_params(labelbottom=False)

        except:
            axs[r, c].remove()
        
plt.show()

In [ ]:
for col in numeric_col:
    lwb, upb = getOutliersBounds(df_main, col)
    outliers = df_main.filter((pl.col(col) > upb) | (pl.col(col) < lwb))
    print(f'{col}, high bound = {upb:,} ,{outliers.height:,} outliers of {df_main.height:,} values, {(outliers.height/df_main.height)*100:.2f} %')

In [ ]:
outliers_cols = [
    c for c in df_main.select(cs.numeric()).columns 
    if df_main[c].median() > 0
]

In [ ]:
bounds = [getOutliersBounds(df_main, col) for col in outliers_cols]

In [ ]:
df_final = df_main.clone()
for col_name, bound in zip(outliers_cols, bounds):
    df_final = df_final.filter(
        (pl.col(col_name) <= bound[1]) &
        (pl.col(col_name) >= bound[0])
    )

In [ ]:
fig, axs = plt.subplots(figsize=(12, 8), ncols=n_cols, nrows=n_rows)

for r in range(n_rows):
    for c in range(n_cols):
        ndx = c + (4 * r) +1
        try:
            data = df_final.select(pl.col(numeric_col[ndx]))
            sns.boxplot(data=data, legend=False, ax=axs[r, c])
            axs[r, c].set_title(numeric_col[ndx])
            axs[r, c].tick_params(labelbottom=False)

        except:
            axs[r, c].remove()
        
plt.show()

In [ ]:
df_final.select(
    cs.numeric()
).describe()

In [ ]:
df_final.write_csv('Datasets/final_credit_train.csv')

## **2.2**
<p style="height:60px;line-height:60px;background-color:navy;padding-left:12px;font-size:32px;font-family:serif;font-weight:bold;color:white;border:1px outset gold;border-radius:8px;">
    Analytics
</p>

### **2.2.1**
<p style="height:50px;line-height:50px;background-color:yellow;padding-left:12px;font-size:28px;font-family:serif;font-weight:bold;color:black;border:1px outset gold;border-radius:6px;">
    Quantitative
</p>

**NOTE**

> Because i have 4GB of RAM, i will get two samples of 5000 observations and analysis it.
>
> one sample before removing outliers and one sample after removing outliers.

In [ ]:
# after removing outliers
aft_sample = df_final.sample(n=5000, shuffle=True, seed=42)

# before removing outliers
bef_sample = df_main.sample(n=5000, shuffle=True, seed=22)

```python
# Correlation & Relationships

corr_matrix = df_main.select(
    cs.numeric()
).corr()

sns.heatmap(
    data=corr_matrix, 
    annot=True, 
    fmt='.2f', 
    cmap='Reds', 
    xticklabels=corr_matrix.columns, 
    yticklabels=corr_matrix.columns
)

plt.show()
```

#### Sample before removing outliers

---

**Distribution**

---

In [ ]:
qnt_cols = bef_sample.select(cs.numeric()).columns
n_cols = 3
n_rows = int(np.ceil(len(qnt_cols) / n_cols))

fig, axs = plt.subplots(figsize=(12, 8), ncols=n_cols, nrows=n_rows)

for r in range(n_rows):
    for c in range(n_cols):
        ndx = c + (3 * r) +1
        try:
            data = bef_sample.select(pl.col(qnt_cols[ndx]))
            sns.histplot(data=data, bins=10, kde=True, ax=axs[r, c], legend=False)
            axs[r, c].set_title(qnt_cols[ndx])
            axs[r, c].tick_params(labelbottom=False, labelleft=False)

        except:
            axs[r, c].remove()
        
plt.show()

**Relations**

---

In [ ]:
pd_data = bef_sample.select(cs.numeric()).to_pandas()

sns.pairplot(data=pd_data, hue="Result", corner=True)

plt.show()

**Correlation**

---

In [ ]:
corr = bef_sample.select(cs.numeric()).corr().to_pandas()

sns.heatmap(
    data=corr,
    annot=True, 
    fmt='.2f', 
    xticklabels=corr.columns, 
    yticklabels=corr.columns,
    cmap='Reds'
)

plt.show()

#### Sample after removing outliers

---

**Distribution**

---

In [ ]:
qnt_cols = aft_sample.select(cs.numeric()).columns
n_cols   = 3
n_rows   = int(np.ceil(len(qnt_cols) / n_cols))

fig, axs = plt.subplots(figsize=(12, 8), ncols=n_cols, nrows=n_rows)

for r in range(n_rows):
    for c in range(n_cols):
        ndx = c + (3 * r) +1
        try:
            data = aft_sample.select(pl.col(qnt_cols[ndx]))
            sns.histplot(data=data, bins=10, kde=True, ax=axs[r, c], legend=False)
            axs[r, c].set_title(qnt_cols[ndx])
            axs[r, c].tick_params(labelbottom=False, labelleft=False)

        except:
            axs[r, c].remove()
        
plt.show()

**Relations**

---

In [ ]:
pd_data = bef_sample.select(cs.numeric()).to_pandas()

sns.pairplot(data=pd_data, hue="Result", corner=True)

plt.show()

**Correlation**

---

In [ ]:
corr = aft_sample.select(cs.numeric()).corr().to_pandas()

sns.heatmap(
    data=corr,
    annot=True, 
    fmt='.2f', 
    xticklabels=corr.columns, 
    yticklabels=corr.columns,
    cmap='Reds'
)

plt.show()

## 2.3
<p style="height:50px;line-height:50px;background-color:purple;padding-left:12px;font-size:28px;font-family:serif;font-weight:bold;color:white;border:1px outset gold;border-radius:6px;">
    CONCLUSION
</p>

There are many outliers in some variables; removing them would significantly alter the analysis results and data measurements, while retaining them would cause bias and inaccuracy in the measurements and analysis.

Therefore, stop the study here.

# **Kaggle : [Tarek Ghajary](www.kaggle.com/tarekghajary) | GitHub : [CHESyrian](www.github.com/CHESyrian) | Linkedin : [Tarek Ghajary](https://www.linkedin.com/in/tarek-ghajary-95ab0b195/)**
<p style="height:240px;line-height:100px;font-size:64px;background-color:cyan;color:white;padding:20px auto;text-align:center;border:2px outset gold;border-radius:8px;">
    Created by:<br/> <code style="border:1px outset gold;border-radius:10px;">Tarek Ghajary</code>
</p>